# 机械臂仿真：运行一次，看看相机画面

在 Google Colab 中上传这个 `.ipynb` 文件，使用普通 **CPU** 运行时，
按顺序运行下面的代码单元。无需在自己的 Mac 上安装机械臂软件。

本文件已包含本次演示脚本，无需连接 GitHub 或 Google Drive。
第一次运行会下载 Python 和依赖；仿真在当前云端机器运行。
完成后，下方会直接显示四格动图和截图。

**这次做什么：** 用 PyBullet 自带的 Franka Panda 模型执行预先设置的关节运动，
对比固定相机、原始手腕相机、手腕相机局部 x 方向 +5 cm、局部 y 方向 +15°。
四幅画面来自同一时刻的场景状态。

**如何解释：** 这用于观察相机视角、目标可见性和遮挡；
没有训练 VLA，也没有把机械臂运动当成抓取成功。
师兄提供的 RP 仅是外部参考，本笔记本不包含其原文。

本笔记本保存时未执行，实际是否成功以你的运行输出为准。


## 1. 准备运行环境

这里使用 Python 3.11 的独立环境，避免 Colab 自身 Python 版本改变后，
PyBullet 需要从源码编译。安装只作用于本次云端运行时。


In [ ]:
import importlib.util
from pathlib import Path
import platform
import subprocess
import sys

if platform.system() != "Linux":
    raise RuntimeError("请把这个文件上传到 Google Colab 的 Linux CPU 运行时，再运行本单元。")

WORK = Path.cwd() / "robot_arm_demo"
WORK.mkdir(exist_ok=True)
if importlib.util.find_spec("uv") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "uv"], check=True)

UV = [sys.executable, "-m", "uv"]
subprocess.run(UV + ["python", "install", "3.11"], check=True)
VENV = WORK / ".venv-py311"
PYTHON = VENV / "bin" / "python"
if not PYTHON.exists():
    subprocess.run(UV + ["venv", "--python", "3.11", str(VENV)], check=True)
subprocess.run(UV + [
    "pip", "install", "--python", str(PYTHON), "--only-binary", ":all:",
    "pybullet==3.2.7", "numpy==2.4.6", "Pillow==13.0.0",
], check=True)
subprocess.run([str(PYTHON), "--version"], check=True)
print("环境已准备。下面将写入演示脚本。")


## 2. 写入演示脚本

运行下面这一格即可。它包含仓库 `scripts/run_demo.py` 的完整副本，
并检查内容哈希。重复运行会更新这份脚本；后续每次仿真会创建新的结果文件夹。


In [ ]:
import hashlib
DEMO_SOURCE = '#!/usr/bin/env python3\n"""Run a small, real PyBullet camera-observation demo on a CPU.\n\nUses the Franka Panda model shipped with pybullet_data. Motion is prescribed\njoint-position control, not a learned policy or a successful grasping trial.\nThe wrist camera is aimed once at setup and then remains rigidly attached.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nfrom datetime import datetime, timezone\nimport hashlib\nfrom importlib import metadata as package_metadata\nimport json\nimport math\nfrom pathlib import Path\nimport platform\nimport shlex\nimport subprocess\nimport sys\nimport time\n\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageFont\nimport pybullet as p\nimport pybullet_data\n\n\nCAMERA_NAMES = ("external", "wrist_baseline", "wrist_tx_plus_5cm", "wrist_ry_plus_15deg")\nPANEL_LABELS = ("Fixed external camera", "Wrist camera: original mount",\n                "Wrist mount: local x +5 cm", "Wrist mount: local y +15 deg")\n\n\ndef transform(position, quaternion):\n    matrix = np.eye(4)\n    matrix[:3, :3] = np.asarray(p.getMatrixFromQuaternion(quaternion)).reshape(3, 3)\n    matrix[:3, 3] = position\n    return matrix\n\n\ndef look_at_optical(eye, target):\n    """World-from-optical pose: optical x right, y down, z forward."""\n    eye, target = np.asarray(eye, dtype=float), np.asarray(target, dtype=float)\n    forward = target - eye\n    forward /= np.linalg.norm(forward)\n    up = np.array([0.0, 0.0, 1.0])\n    if abs(np.dot(forward, up)) > 0.95:\n        up = np.array([0.0, 1.0, 0.0])\n    right = np.cross(forward, up)\n    right /= np.linalg.norm(right)\n    down = np.cross(forward, right)\n    matrix = np.eye(4)\n    matrix[:3, :3] = np.column_stack([right, down, forward])\n    matrix[:3, 3] = eye\n    return matrix\n\n\ndef box(half_extents, position, color, mass=0.0):\n    shape = p.createCollisionShape(p.GEOM_BOX, halfExtents=half_extents)\n    visual = p.createVisualShape(p.GEOM_BOX, halfExtents=half_extents, rgbaColor=color)\n    return p.createMultiBody(baseMass=mass, baseCollisionShapeIndex=shape,\n                             baseVisualShapeIndex=visual, basePosition=position)\n\n\ndef get_link_transform(robot, link):\n    state = p.getLinkState(robot, link, computeForwardKinematics=True)\n    # [4:6] are the world URDF link frame, not the centre-of-mass frame.\n    return transform(state[4], state[5])\n\n\ndef capture(camera_pose, projection, width, height, target_ids):\n    eye = camera_pose[:3, 3]\n    forward = camera_pose[:3, 2]\n    # OpenGL view up is negative optical y.\n    up = -camera_pose[:3, 1]\n    view = p.computeViewMatrix(eye.tolist(), (eye + forward).tolist(), up.tolist())\n    result = p.getCameraImage(width, height, viewMatrix=view,\n                             projectionMatrix=projection,\n                             renderer=p.ER_TINY_RENDERER,\n                             flags=p.ER_SEGMENTATION_MASK_OBJECT_AND_LINKINDEX,\n                             lightDirection=[-1, -1, 3], shadow=1)\n    pixels = np.asarray(result[2], dtype=np.uint8).reshape(height, width, 4)\n    segmentation = np.asarray(result[4], dtype=np.int64).reshape(height, width)\n    object_ids = segmentation & ((1 << 24) - 1)\n    visible = {name: int(np.count_nonzero((segmentation >= 0) & (object_ids == body_id)))\n               for name, body_id in target_ids.items()}\n    return Image.fromarray(pixels[:, :, :3]), visible\n\n\ndef font(size):\n    try:\n        return ImageFont.load_default(size=size)\n    except TypeError:\n        return ImageFont.load_default()\n\n\ndef comparison(images, frame_index, count, simulated_time):\n    width, height = images[0].size\n    label_height, footer_height = 32, 44\n    canvas = Image.new("RGB", (width * 2, (height + label_height) * 2 + footer_height), "#101c2d")\n    draw = ImageDraw.Draw(canvas)\n    for index, (picture, label) in enumerate(zip(images, PANEL_LABELS)):\n        x = (index % 2) * width\n        y = (index // 2) * (height + label_height)\n        canvas.paste(picture, (x, y + label_height))\n        draw.text((x + 10, y + 8), label, font=font(14), fill="#f4f7fc")\n    y = (height + label_height) * 2\n    draw.text((10, y + 5), f"Same state in all four views | frame {frame_index + 1}/{count} | sim t={simulated_time:.2f}s",\n              font=font(13), fill="#f4f7fc")\n    draw.text((10, y + 24), "PyBullet CPU rendering | Prescribed motion; no VLA or grasp evaluation",\n              font=font(12), fill="#bfcadd")\n    return canvas\n\n\ndef version(name):\n    try:\n        return package_metadata.version(name)\n    except package_metadata.PackageNotFoundError:\n        return "unknown"\n\n\ndef git_state(root):\n    try:\n        head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=root, text=True, stderr=subprocess.DEVNULL).strip()\n        dirty = bool(subprocess.check_output(["git", "status", "--porcelain"], cwd=root, text=True))\n        return {"head": head, "working_tree_dirty": dirty,\n                "note": "The script SHA256 below identifies executed code even if this tree was dirty."}\n    except (OSError, subprocess.CalledProcessError):\n        return {"head": None, "working_tree_dirty": None}\n\n\ndef run(args):\n    started = datetime.now(timezone.utc)\n    run_dir = args.output / started.strftime("run_%Y%m%dT%H%M%S_%fZ")\n    run_dir.mkdir(parents=True, exist_ok=False)\n    script = Path(__file__).resolve()\n    script_hash = hashlib.sha256(script.read_bytes()).hexdigest()\n    vcs = git_state(script.parent.parent)\n    client = p.connect(p.GUI if args.gui else p.DIRECT)\n    if client < 0:\n        raise RuntimeError("PyBullet connection failed. Try again without --gui for CPU image output.")\n    physics_hz = 240\n    steps_per_frame = max(1, round(physics_hz / args.fps))\n    # GIF stores duration in centiseconds. Record the value that is really saved.\n    gif_duration_ms = max(10, round(1000 / args.fps / 10) * 10)\n    try:\n        p.setAdditionalSearchPath(pybullet_data.getDataPath())\n        p.setGravity(0, 0, -9.81)\n        p.setTimeStep(1.0 / physics_hz)\n        p.setPhysicsEngineParameter(numSolverIterations=80, deterministicOverlappingPairs=1)\n        p.loadURDF("plane.urdf")\n        box([0.7, 0.5, 0.035], [0.35, 0, 0.60], [0.72, 0.76, 0.80, 1])\n        for x in [-0.23, 0.93]:\n            for y in [-0.40, 0.40]:\n                box([0.035, 0.035, 0.285], [x, y, 0.285], [0.24, 0.29, 0.36, 1])\n        robot = p.loadURDF("franka_panda/panda.urdf", [-0.25, 0, 0.636], useFixedBase=True)\n        joints = [index for index in range(p.getNumJoints(robot))\n                  if p.getJointInfo(robot, index)[1].decode().startswith("panda_joint")\n                  and p.getJointInfo(robot, index)[2] == p.JOINT_REVOLUTE]\n        links = {p.getJointInfo(robot, index)[12].decode(): index\n                 for index in range(p.getNumJoints(robot))}\n        hand_link = links["panda_hand"]\n        initial_joints = np.array([0.0, -0.45, 0.0, -2.10, 0.0, 1.80, 0.78])\n        for joint, value in zip(joints, initial_joints):\n            p.resetJointState(robot, joint, float(value))\n        for index in range(p.getNumJoints(robot)):\n            if p.getJointInfo(robot, index)[2] == p.JOINT_PRISMATIC:\n                p.resetJointState(robot, index, 0.04)\n                p.setJointMotorControl2(robot, index, p.POSITION_CONTROL, targetPosition=0.04, force=20)\n        targets = {\n            "red_cube": box([0.030, 0.030, 0.030], [0.30, -0.08, 0.666], [0.90, 0.14, 0.16, 1], 0.05),\n            "blue_cube": box([0.025, 0.025, 0.025], [0.20, 0.17, 0.661], [0.12, 0.40, 0.95, 1], 0.05),\n            "yellow_cube": box([0.035, 0.035, 0.035], [0.45, 0.13, 0.671], [0.98, 0.73, 0.10, 1], 0.05),\n        }\n        p.setJointMotorControlArray(robot, joints, p.POSITION_CONTROL,\n                                    targetPositions=initial_joints.tolist(), forces=[80] * 7)\n        for _ in range(120):\n            p.stepSimulation()\n        initial_hand = get_link_transform(robot, hand_link)\n        # Aim once, then freeze the E<-C mounting transform for the entire run.\n        eye = (initial_hand @ np.array([0.08, -0.06, 0.025, 1.0]))[:3]\n        initial_camera = look_at_optical(eye, [0.31, 0.06, 0.65])\n        mount = np.linalg.inv(initial_hand) @ initial_camera\n        delta_translation = np.eye(4)\n        delta_translation[0, 3] = 0.05\n        angle = math.radians(15)\n        delta_rotation = np.eye(4)\n        delta_rotation[:3, :3] = [[math.cos(angle), 0, math.sin(angle)], [0, 1, 0],\n                                  [-math.sin(angle), 0, math.cos(angle)]]\n        external = look_at_optical([1.22, -1.32, 1.46], [0.20, 0, 0.89])\n        projection = p.computeProjectionMatrixFOV(fov=60, aspect=args.width / args.height,\n                                                  nearVal=0.02, farVal=4.0)\n        if args.gui:\n            p.resetDebugVisualizerCamera(cameraDistance=1.75, cameraYaw=42, cameraPitch=-28,\n                                          cameraTargetPosition=[0.2, 0, 0.9])\n            print("GUI open. Close the PyBullet window to exit; images are also saved to the run folder.")\n        frames, records = [], []\n        snapshot_index = args.frames // 3\n        for frame_index in range(args.frames):\n            if not p.isConnected(client):\n                raise RuntimeError("GUI closed before the run completed; no completed-run claim was saved.")\n            frame_started = time.monotonic()\n            phase = 2 * math.pi * frame_index / (args.frames - 1)\n            offset = np.array([0.20 * math.sin(phase), 0.13 * math.sin(phase),\n                               0.06 * math.sin(phase), 0.10 * math.sin(phase + 0.4),\n                               0.06 * math.sin(phase), 0.10 * math.sin(phase), 0.07 * math.sin(phase)])\n            p.setJointMotorControlArray(robot, joints, p.POSITION_CONTROL,\n                                        targetPositions=(initial_joints + offset).tolist(), forces=[80] * 7)\n            for _ in range(steps_per_frame):\n                p.stepSimulation()\n            world_hand = get_link_transform(robot, hand_link)\n            wrist = world_hand @ mount\n            poses = [external, wrist, wrist @ delta_translation, wrist @ delta_rotation]\n            pictures, visibility = [], {}\n            # No simulation step occurs between these captures: they share one state.\n            for name, pose in zip(CAMERA_NAMES, poses):\n                picture, counts = capture(pose, projection, args.width, args.height, targets)\n                pictures.append(picture)\n                visibility[name] = counts\n            simulated_time = (120 + (frame_index + 1) * steps_per_frame) / physics_hz\n            combined = comparison(pictures, frame_index, args.frames, simulated_time)\n            frames.append(combined)\n            records.append({"frame": frame_index, "simulation_time_s": simulated_time,\n                            "joint_positions_rad": [p.getJointState(robot, j)[0] for j in joints],\n                            "world_from_hand": world_hand.tolist(),\n                            "world_from_optical_camera": {name: pose.tolist() for name, pose in zip(CAMERA_NAMES, poses)},\n                            "visible_target_pixels": visibility})\n            if frame_index == snapshot_index:\n                combined.save(run_dir / "comparison.png")\n                for name, picture in zip(CAMERA_NAMES, pictures):\n                    picture.save(run_dir / f"{name}.png")\n            if args.gui:\n                time.sleep(max(0, 1 / args.fps - (time.monotonic() - frame_started)))\n        frames[0].save(run_dir / "comparison.gif", save_all=True, append_images=frames[1:],\n                       duration=gif_duration_ms, loop=0, optimize=False, disposal=2)\n        fy = (args.height / 2) / math.tan(math.radians(60) / 2)\n        report = {\n            "status": "completed", "started_at_utc": started.isoformat(),\n            "completed_at_utc": datetime.now(timezone.utc).isoformat(),\n            "scope": "CPU physics simulation and paired camera observations with prescribed joint motion.",\n            "not_evaluated": ["learned VLA policy", "grasping success", "task success rate", "research hypothesis"],\n            "runtime": {"python": platform.python_version(), "platform": platform.platform(),\n                        "pybullet": version("pybullet"), "numpy": version("numpy"), "Pillow": version("Pillow"),\n                        "connection": "GUI" if args.gui else "DIRECT", "renderer": "ER_TINY_RENDERER"},\n            "command": shlex.join([sys.executable, *sys.argv]), "script_sha256": script_hash,\n            "git": vcs, "seed": args.seed,\n            "seed_note": "Scene and trajectory are deterministic; no random sampling is performed.",\n            "robot_model": "pybullet_data/franka_panda/panda.urdf", "wrist_link": "panda_hand",\n            "control": "POSITION_CONTROL with sinusoidal prescribed joint targets",\n            "physics_hz": physics_hz, "warmup_steps": 120, "steps_per_frame": steps_per_frame,\n            "frame_count": len(frames), "requested_playback_fps": args.fps,\n            "gif_frame_duration_ms": gif_duration_ms,\n            "actual_gif_playback_fps": 1000 / gif_duration_ms, "snapshot_frame": snapshot_index,\n            "camera": {"width": args.width, "height": args.height, "vertical_fov_deg": 60,\n                       "near_m": 0.02, "far_m": 4.0,\n                       "intrinsics_pixel_centres": [[fy, 0, (args.width - 1) / 2],\n                                                     [0, fy, (args.height - 1) / 2], [0, 0, 1]],\n                       "convention": "Column vectors; T_A_B maps B to A; metres; optical x right, y down, z forward.",\n                       "view_conversion": "OpenGL view forward = optical +z; up = optical -y.",\n                       "hand_from_baseline_camera": mount.tolist(),\n                       "translation_delta_local": delta_translation.tolist(),\n                       "rotation_delta_local": delta_rotation.tolist(),\n                       "composition": "T_world_C = T_world_hand @ T_hand_C @ Delta_local",\n                       "synchronization": "All four views rendered without intervening physics steps."},\n            "outputs": ["comparison.gif", "comparison.png", *[f"{name}.png" for name in CAMERA_NAMES]],\n            "frames": records,\n        }\n        (run_dir / "metadata.json").write_text(json.dumps(report, indent=2) + "\\n", encoding="utf-8")\n        print(f"Completed {len(frames)} simulated observation frames.")\n        print(f"Output: {run_dir.resolve()}")\n        print("Open comparison.gif for motion, comparison.png for the four-view snapshot, metadata.json for evidence.")\n        if args.gui and args.keep_open:\n            print("Run saved. Close the PyBullet window to finish.")\n            while p.isConnected(client):\n                time.sleep(0.1)\n        return run_dir\n    except Exception as exc:\n        (run_dir / "error.json").write_text(json.dumps({"status": "failed", "error": str(exc),\n                                                        "script_sha256": script_hash}, indent=2) + "\\n")\n        raise\n    finally:\n        if p.isConnected(client):\n            p.disconnect(client)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--output", type=Path, default=Path("results/demo"), help="Parent folder; every run gets a new timestamped directory.")\n    parser.add_argument("--gui", action="store_true", help="Also open the local PyBullet 3D window.")\n    parser.add_argument("--keep-open", action="store_true", help="With --gui, keep the window open after saving.")\n    parser.add_argument("--frames", type=int, default=64)\n    parser.add_argument("--fps", type=int, default=12)\n    parser.add_argument("--width", type=int, default=360)\n    parser.add_argument("--height", type=int, default=280)\n    parser.add_argument("--seed", type=int, default=7, help="Recorded identifier; this demo does not sample randomness.")\n    args = parser.parse_args()\n    if not (2 <= args.frames <= 600 and 1 <= args.fps <= 60 and 160 <= args.width <= 1920 and 120 <= args.height <= 1080):\n        parser.error("Require frames 2..600, fps 1..60, width 160..1920 and height 120..1080.")\n    if args.keep_open and not args.gui:\n        parser.error("--keep-open requires --gui.")\n    run(args)\n\n\nif __name__ == "__main__":\n    main()\n'
EXPECTED_SHA256 = '9270a2bf379b1c81ab276488be25d6ecd5c6c8474bfdeaa43578f445aff4eb4c'
assert hashlib.sha256(DEMO_SOURCE.encode('utf-8')).hexdigest() == EXPECTED_SHA256
SCRIPT = WORK / 'scripts' / 'run_demo.py'
SCRIPT.parent.mkdir(exist_ok=True)
SCRIPT.write_text(DEMO_SOURCE, encoding='utf-8')
print('脚本已写入：', SCRIPT)
print('SHA-256：', EXPECTED_SHA256)


## 3. 跑一次仿真

首次直接使用默认设置。`FRAMES` 决定采集帧数，`FPS` 决定动图播放速度。
保留小分辨率即可；这次不需要 GPU。

等单元完成后，会显示本次结果的文件夹。若出现错误，保留完整错误文字，
不要把安装完成或出现部分文件当成仿真成功。


In [ ]:
FRAMES = 64
FPS = 12
WIDTH, HEIGHT = 360, 280
SEED = 7  # 本演示不随机采样；这个数字作为记录标识。

RESULTS = WORK / "results" / "demo"
RESULTS.mkdir(parents=True, exist_ok=True)
before = set(RESULTS.glob("*/metadata.json"))
command = [
    str(PYTHON), str(SCRIPT), "--output", str(RESULTS),
    "--frames", str(FRAMES), "--fps", str(FPS),
    "--width", str(WIDTH), "--height", str(HEIGHT), "--seed", str(SEED),
]
print("正在运行：", " ".join(command), flush=True)
subprocess.run(command, cwd=WORK, check=True)
created = set(RESULTS.glob("*/metadata.json")) - before
if len(created) != 1:
    raise RuntimeError("没有找到唯一的新运行记录，请查看上方输出及结果目录。")
RUN_DIR = created.pop().parent
print("本次结果：", RUN_DIR)


## 4. 看效果

- **左上：** 固定外部相机，可以看到整个机械臂运动。
- **右上：** 原始手腕相机，随末端一起移动。
- **左下：** 在原始安装上，沿相机局部 x 轴平移 +5 cm。
- **右下：** 在原始安装上，绕相机局部 y 轴旋转 +15°。

先看方块是否仍在画面里、位置怎样改变，以及手臂有没有挡住相机。
这些是图像观察，不是模型性能分数。


In [ ]:
import json
from IPython.display import Image, display

display(Image(filename=str(RUN_DIR / "comparison.gif")))
display(Image(filename=str(RUN_DIR / "comparison.png")))
record = json.loads((RUN_DIR / "metadata.json").read_text(encoding="utf-8"))
print("结果文件：")
for path in sorted(RUN_DIR.iterdir()):
    print(" -", path.name)
print("metadata.json 保存软件版本、运行信息和逐帧几何记录。")


## 5. 可选：保存这次结果

云端运行时可能被回收。想把结果留在自己电脑上时，将下方
`DOWNLOAD_RESULTS` 改成 `True`，再运行这一格。
它只打包本次结果，不包含 Python 环境或第三方模型文件。


In [ ]:
DOWNLOAD_RESULTS = False

if DOWNLOAD_RESULTS:
    import shutil
    archive = shutil.make_archive(str(WORK / RUN_DIR.name), "zip", root_dir=RUN_DIR)
    try:
        from google.colab import files
    except ImportError:
        print("结果压缩包：", archive)
    else:
        files.download(archive)
else:
    print("本次没有触发下载；上方动图和截图仍可查看。")
